# 00 - Environment Setup for dlt8 (ECUSTFD Food Calorie Estimation)

This notebook recreates a clean, reproducible environment for the **dlt8** project:
Python version, required packages, CUDA/GPU check, project paths, and module import sanity check.

Re-run this notebook whenever you move the project to a new machine or after a fresh `pip install`.

## 1. Paths & layout

In [3]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

# Project root = parent of 'src' (notebook lives in src/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT      = PROJECT_ROOT / 'data'
RAW_ECUSTFD    = DATA_ROOT / 'raw' / 'ECUSTFD'
PROCESSED_ROOT = DATA_ROOT / 'processed'
LOGS_ROOT      = PROJECT_ROOT / 'logs'
MODELS_ROOT    = PROJECT_ROOT / 'models'
OUTPUTS_ROOT   = PROJECT_ROOT / 'outputs'
FASTER_RCNN    = PROJECT_ROOT / 'ECUSTFD' / 'faster_rcnn'

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"RAW_ECUSTFD  : {RAW_ECUSTFD}  exists={RAW_ECUSTFD.exists()}")
print(f"FASTER_RCNN  : {FASTER_RCNN}  exists={FASTER_RCNN.exists()}")
print(f"LOGS_ROOT    : {LOGS_ROOT}  exists={LOGS_ROOT.exists()}")

for p in [DATA_ROOT, RAW_ECUSTFD, LOGS_ROOT, MODELS_ROOT, OUTPUTS_ROOT, FASTER_RCNN]:
    p.mkdir(parents=True, exist_ok=True)

Python       : 3.14.4
Platform     : Windows-11-10.0.26200-SP0
PROJECT_ROOT : E:\AI_Research\dlt8
RAW_ECUSTFD  : E:\AI_Research\dlt8\data\raw\ECUSTFD  exists=True
FASTER_RCNN  : E:\AI_Research\dlt8\ECUSTFD\faster_rcnn  exists=True
LOGS_ROOT    : E:\AI_Research\dlt8\logs  exists=True


## 2. Python & pip version sanity

In [4]:
assert sys.version_info >= (3, 10), "Python >= 3.10 required (project uses PEP 604 unions, match-case, etc.)."
print("Python version OK")

print("pip version:", subprocess.check_output([sys.executable, '-m', 'pip', '--version']).decode().strip())

Python version OK
pip version: pip 26.2 from C:\Users\Admin\AppData\Roaming\Python\Python314\site-packages\pip (python 3.14)


## 3. Install dependencies from `requirements.txt`

We install everything the project actually needs. The list is curated from `requirements.txt` at the repo root. If you need to skip the heavy install (PyTorch + Ultralytics), set `SKIP_HEAVY = True` below.

In [5]:
SKIP_HEAVY = False  # Set True to skip torch / ultralytics / segment-anything (already installed on this machine)

req_file = PROJECT_ROOT / 'requirements.txt'
print(f"Using requirements file: {req_file} (exists={req_file.exists()})")
print(req_file.read_text(encoding='utf-8'))

Using requirements file: E:\AI_Research\dlt8\requirements.txt (exists=True)
# Food Volume & Calorie Estimation — Python Dependencies
# Install: pip install -r requirements.txt

# Core
numpy>=1.24.0
pandas>=2.0.0
scipy>=1.10.0

# Dataset parsing
xlrd>=2.0.1  # Read .xls files (density.xls)
openpyxl>=3.1.0  # Alternative for .xlsx

# Image processing
opencv-python>=4.8.0
Pillow>=10.0.0
imageio>=2.30.0

# Deep Learning
torch>=2.0.0
torchvision>=0.15.0

# YOLO
ultralytics>=8.2.0  # YOLO13 / YOLOv8 framework

# SAM (Segment Anything Model)
segment-anything==1.0
# Download checkpoint: sam_vit_b_01ec64.pth from https://github.com/facebookresearch/segment-anything
# Save to: models/sam/sam_vit_b_01ec64.pth
# (model weights are committed in-repo; reviewer does not need to re-download)

# Depth Estimation (MiDaS)
# pip install timm

# XAI
lime>=0.2.0.0

# Visualization
matplotlib>=3.7.0
seaborn>=0.12.0

# Jupyter
jupyter>=1.0.0
ipykernel>=6.25.0
nbformat>=5.9.0

# Progress bars
tqdm>=4.65.0

# 3

In [6]:
%pip install --upgrade pip

req_abs = str(PROJECT_ROOT / 'requirements.txt')

if SKIP_HEAVY:
    %pip install -r $req_abs --ignore-requires-python
    # torch and ultralytics will be installed separately if you un-skip
else:
    # Install PyTorch first (CUDA 12.1 build for broadest compatibility on Windows).
    # If you have a different CUDA version, change the index URL accordingly.
    import torch
    try:
        torch_ver = torch.__version__
        print(f"torch already installed: {torch_ver}")
    except ImportError:
        print("Installing torch (CUDA 12.1) ...")
        %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
    %pip install -r $req_abs --ignore-requires-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
torch already installed: 2.13.0+cu130
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 4. GPU / CUDA check

In [7]:
import torch
print(f"torch        : {torch.__version__}")
print(f"CUDA avail.  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version : {torch.version.cuda}")
    print(f"Device count : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  [{i}] {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: no CUDA GPU detected. Training will fall back to CPU (very slow). ")

torch        : 2.13.0+cu130
CUDA avail.  : True
CUDA version : 13.0
Device count : 1
  [0] NVIDIA GeForce RTX 4050 Laptop GPU


## 5. Verify the core libs are importable

In [8]:
imports = [
    'numpy', 'pandas', 'scipy',
    'cv2', 'PIL', 'imageio',
    'matplotlib', 'seaborn', 'tqdm',
    'xlrd', 'openpyxl',
]
heavy = ['torch', 'torchvision', 'ultralytics', 'segment_anything', 'timm']

missing = []
for mod in imports:
    try:
        __import__(mod); print(f"  [OK] {mod}")
    except Exception as e:
        print(f"  [FAIL] {mod}: {e}"); missing.append(mod)

if not SKIP_HEAVY:
    for mod in heavy:
        try:
            __import__(mod); print(f"  [OK] {mod}")
        except Exception as e:
            print(f"  [FAIL] {mod}: {e}"); missing.append(mod)
else:
    print("(Heavy import check skipped due to SKIP_HEAVY=True)")

if missing:
    print(f"\nMissing modules: {missing}")
else:
    print("\nAll imports succeeded.")

  [OK] numpy
  [OK] pandas
  [OK] scipy
  [OK] cv2
  [OK] PIL
  [OK] imageio
  [OK] matplotlib
  [OK] seaborn
  [OK] tqdm
  [OK] xlrd
  [OK] openpyxl
  [OK] torch
  [OK] torchvision
  [OK] ultralytics
  [OK] segment_anything
  [OK] timm

All imports succeeded.


## 6. Make `src` importable as a package

In [9]:
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print(sys.path[:3], '...')

try:
    import config, constants
    print(f"config.PROJECT_ROOT   = {config.PROJECT_ROOT}")
    print(f"config.RAW_DATA_ROOT  = {config.RAW_DATA_ROOT}")
    print(f"constants.FOOD_CLASSES ({len(constants.FOOD_CLASSES)}): {constants.FOOD_CLASSES}")
except Exception as e:
    print(f"Import failed: {e}")
    print("If you launched Jupyter from inside src/, change cwd to the repo root and re-run.")

['E:\\AI_Research\\dlt8\\src', 'c:\\Python314\\python314.zip', 'c:\\Python314\\DLLs'] ...
config.PROJECT_ROOT   = E:\AI_Research\dlt8
config.RAW_DATA_ROOT  = E:\AI_Research\dlt8\data\raw\ECUSTFD
constants.FOOD_CLASSES (19): ['apple', 'banana', 'bread', 'bun', 'doughnut', 'egg', 'fired_dough_twist', 'grape', 'lemon', 'litchi', 'mango', 'mooncake', 'orange', 'peach', 'pear', 'plum', 'qiwi', 'sachima', 'tomato']


## 7. Final environment report

In [11]:
report = {
    "python"  : sys.version.split()[0],
    "platform": platform.platform(),
    "cwd"     : os.getcwd(),
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "RAW_ECUSTFD" : str(RAW_ECUSTFD),
    "FASTER_RCNN" : str(FASTER_RCNN),
    "LOGS_ROOT"   : str(LOGS_ROOT),
    "torch"       : torch.__version__,
    "cuda_avail"  : torch.cuda.is_available(),
    "cuda_device" : torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
for k, v in report.items():
    print(f"{k:13s}: {v}")

log_path = LOGS_ROOT / 'setup_env_report.txt'
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text("\n".join(f"{k}={v}" for k, v in report.items()), encoding='utf-8')
print(f"\nReport saved to {log_path}")

python       : 3.14.4
platform     : Windows-11-10.0.26200-SP0
cwd          : e:\AI_Research\dlt8\src
PROJECT_ROOT : E:\AI_Research\dlt8
RAW_ECUSTFD  : E:\AI_Research\dlt8\data\raw\ECUSTFD
FASTER_RCNN  : E:\AI_Research\dlt8\ECUSTFD\faster_rcnn
LOGS_ROOT    : E:\AI_Research\dlt8\logs
torch        : 2.13.0+cu130
cuda_avail   : True
cuda_device  : NVIDIA GeForce RTX 4050 Laptop GPU

Report saved to E:\AI_Research\dlt8\logs\setup_env_report.txt
